# 01 — SimCLR Training on CIFAR-10

Train SimCLR self-supervised model on CIFAR-10 (Step 1 of TPC_RP).

**Architecture:** ResNet-18 (CIFAR-adapted) | **Epochs:** 500 (matching paper) | **LR:** 0.4 cosine

In [ ]:
# Colab setup (uncomment if needed)
# !pip install torch torchvision scikit-learn matplotlib tqdm

import sys, os
if os.path.basename(os.getcwd()) == 'notebooks':
    sys.path.insert(0, os.path.dirname(os.getcwd()))
else:
    sys.path.insert(0, os.getcwd())

import torch
from torch.utils.data import DataLoader
from torchvision import datasets

from src.simclr import SimCLR, train_simclr, extract_features
from src.augmentations import SimCLRTransform, get_embedding_transform
from src.utils import get_device, set_seed, MODELS_DIR, DATA_DIR

set_seed(42)
device = get_device()

Using Apple MPS


## 1. Load CIFAR-10 with SimCLR Augmentations

In [ ]:
simclr_transform = SimCLRTransform(image_size=32)
train_dataset_simclr = datasets.CIFAR10(
    root=str(DATA_DIR), train=True, download=True, transform=simclr_transform
)

num_workers = 2 if device.type == 'cuda' else 0
simclr_loader = DataLoader(
    train_dataset_simclr, batch_size=512, shuffle=True,
    num_workers=num_workers, pin_memory=(device.type == 'cuda'), drop_last=True,
)
print(f'Dataset: {len(train_dataset_simclr)} images | Batches: {len(simclr_loader)}')

100%|██████████| 170M/170M [00:11<00:00, 15.3MB/s] 


Dataset: 50000 images | Batches: 97


## 2. Train SimCLR

**Note:** We train for 500 epochs matching the paper. For faster local testing, reduce to 200.

In [ ]:
SIMCLR_EPOCHS = 500
SIMCLR_PATH = str(MODELS_DIR / 'simclr_cifar10.pth')

model = SimCLR(feature_dim=512, projection_dim=128)

losses = train_simclr(
    model=model, dataloader=simclr_loader,
    epochs=SIMCLR_EPOCHS, lr=0.4, weight_decay=1e-4,
    temperature=0.5, device=device, save_path=SIMCLR_PATH,
)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 4))
plt.plot(losses)
plt.xlabel('Epoch'); plt.ylabel('NT-Xent Loss')
plt.title(f'SimCLR Training Loss ({SIMCLR_EPOCHS} epochs)')
plt.grid(True, alpha=0.3); plt.tight_layout()
plt.savefig('../figures/simclr_training_loss.png', dpi=150)
plt.show()

## 3. Extract Features

Extract L2-normalised 512-d features from the encoder for TypiClust.

In [ ]:
# Load model (or use the one just trained)
model = SimCLR(feature_dim=512, projection_dim=128)
model.load_state_dict(torch.load(SIMCLR_PATH, map_location=device, weights_only=True))
model.to(device)

embed_transform = get_embedding_transform()
train_dataset_embed = datasets.CIFAR10(
    root=str(DATA_DIR), train=True, download=False, transform=embed_transform
)
embed_loader = DataLoader(train_dataset_embed, batch_size=256, shuffle=False, num_workers=0)

features, labels = extract_features(model, embed_loader, device=device)
print(f'Features: {features.shape}, Labels: {labels.shape}')

import numpy as np
np.save(str(MODELS_DIR / 'simclr_features.npy'), features.numpy())
np.save(str(MODELS_DIR / 'simclr_labels.npy'), labels.numpy())
print('Features saved.')

## 4. (Optional) t-SNE Visualisation

Uncomment to generate a t-SNE plot of the learned features for the report.

In [ ]:
# from sklearn.manifold import TSNE
# tsne = TSNE(n_components=2, random_state=42, perplexity=30)
# feats_2d = tsne.fit_transform(features.numpy()[:5000])
# plt.figure(figsize=(8, 8))
# plt.scatter(feats_2d[:, 0], feats_2d[:, 1], c=labels.numpy()[:5000], cmap='tab10', s=1, alpha=0.5)
# plt.colorbar(); plt.title('t-SNE of SimCLR Features')
# plt.savefig('../figures/tsne_features.png', dpi=150); plt.show()